# leaf-tensor-condition — worked example 3: Only leaf tensors accumulate .grad — interior tensors do not

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `leaf-tensor-condition`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In PyTorch (and ARENA's hand-built autograd), only leaf tensors with `requires_grad=True` have a `.grad` attribute populated after `.backward()`. Interior tensors (those produced by operations) do not retain gradients by default — their gradient is computed transiently and passed upstream. Calling `.retain_grad()` on an interior tensor is the exception, not the rule.

## Worked solution

Step 1: Build a simple two-leaf graph in PyTorch: `a = torch.tensor([3.0], requires_grad=True)`, `b = torch.tensor([4.0], requires_grad=True)`, `c = a * b`, `loss = c.sum()`.

Step 2: Call `loss.backward()`.

Step 3: Check `a.is_leaf` (True), `b.is_leaf` (True), `c.is_leaf` (False).

Step 4: Print `a.grad` and `b.grad` — both will be populated. Print `c.grad` — it will be `None` because `c` is interior and we did not call `.retain_grad()`.

In [ ]:
import torch as t

a = t.tensor([3.0], requires_grad=True)
b = t.tensor([4.0], requires_grad=True)
c = a * b          # interior tensor
loss = c.sum()

loss.backward()

print('a.is_leaf:', a.is_leaf)   # True
print('b.is_leaf:', b.is_leaf)   # True
print('c.is_leaf:', c.is_leaf)   # False

print('a.grad:', a.grad)         # tensor([4.])
print('b.grad:', b.grad)         # tensor([3.])
print('c.grad:', c.grad)         # None (interior, no retain_grad)

assert a.is_leaf and b.is_leaf
assert not c.is_leaf
assert a.grad is not None
assert b.grad is not None
assert c.grad is None

# With retain_grad: interior can accumulate too
a2 = t.tensor([3.0], requires_grad=True)
b2 = t.tensor([4.0], requires_grad=True)
c2 = a2 * b2
c2.retain_grad()   # opt-in
c2.sum().backward()
print('c2.grad with retain_grad:', c2.grad)  # tensor([1.])